# 08 · Tracker checks: silver, coverage, and the trk_ features

The tracker estimates arterial speeds from bus GPS, about every 10 minutes per segment. Silver turns `speed = -1` (no bus, no estimate) into missing, aggregates each segment to the hour, and places the segment in a community area. This notebook checks:

1. silver against bronze: every reading accounted for;
2. coverage: which areas have segments, and how often they report;
3. the `trk_` features for one month, June 2019;
4. whether congestion relates to crashes at all;
5. point in time: one feature value recomputed by hand from silver.

In [1]:
import os
from datetime import datetime, timedelta

import pandas as pd
import pyspark.sql.functions as F
import yaml

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import main
from utils.gold import target_hours
from utils.tracker import trk_features

with open("config/pipeline.yaml") as f:
    config = yaml.safe_load(f)
spark = main.get_spark(config)

bronze = spark.read.parquet("datamart/bronze/tracker")
silver = spark.read.parquet("datamart/silver/tracker")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/17 09:43:31 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## 1 · Silver against bronze

In [2]:
readings = bronze.count()
no_estimate = bronze.filter(F.col("speed").cast("int") < 0).count()
s = silver.agg(F.count("*").alias("segment_hours"), F.sum("readings").alias("readings"),
               F.sum("readings_with_speed").alias("with_speed"), F.countDistinct("segment_id").alias("segments"),
               F.countDistinct("community_area").alias("areas")).first()
print(f"bronze readings:            {readings:,}")
print(f"  speed = -1 (no estimate): {no_estimate:,} ({no_estimate / readings:.1%})")
print(f"in silver:                  {s.readings:,} readings, {s.with_speed:,} of them with a speed")
print(f"  dropped (segments more than 100 m from every area): {readings - s.readings:,}")
print(f"silver segment-hours:       {s.segment_hours:,}; segments: {s.segments:,} in {s.areas} community areas")

bronze readings:            239,183,199
  speed = -1 (no estimate): 99,404,940 (41.6%)
in silver:                  232,329,631 readings, 136,007,901 of them with a speed
  dropped (segments more than 100 m from every area): 6,853,568
silver segment-hours:       40,638,303; segments: 1,017 in 74 community areas


## 2 · Coverage

A segment-hour has a speed only if a bus passed. How many segments does each area have, and how often do they report?

In [3]:
per_area = (silver.groupBy("community_area").agg(F.countDistinct("segment_id").alias("segments"))
            .toPandas().set_index("community_area")["segments"].sort_values(ascending=False))
print("areas with segments:", len(per_area), "of 77; without:", sorted(set(range(1, 78)) - set(per_area.index)))
print("segments per area: median", per_area.median(), "| most:", per_area.index[0], per_area.iloc[0])
per_area.head(5)

areas with segments: 74 of 77; without: [37, 46, 55]
segments per area: median 11.0 | most: 28 60


community_area
28    60
32    52
8     46
70    36
25    36
Name: segments, dtype: int64

In [4]:
by_year = (silver.groupBy(F.year("hour_start").alias("year"))
           .agg(F.count("*").alias("segment_hours"),
                F.avg((F.col("readings_with_speed") > 0).cast("int")).alias("share_with_speed"),
                F.countDistinct("segment_id").alias("segments"))
           .orderBy("year").toPandas().set_index("year"))
by_year.round(3)

,segment_hours,share_with_speed,segments
year,,,
2018,7291890,0.832,1017
2019,8496018,0.834,1017
2020,8758404,0.827,1017
2021,8209224,0.813,1017
2022,7882767,0.834,1017


## 3 · The trk_ features for June 2019

Built by the same function the feature store will call, for the target hours a June run would write.

In [5]:
targets = target_hours(datetime(2019, 6, 1), datetime(2019, 7, 1), config)
features = trk_features(targets, config, spark).toPandas()
print(f"{len(features):,} rows = 77 areas x {len(targets)} hours")
print("missing congestion index:", f"{features['trk_congestion_index'].isna().mean():.1%}",
      "| missing coverage:", f"{features['trk_coverage'].isna().mean():.1%}")
features[["trk_congestion_index", "trk_coverage"]].describe().round(3)

55,440 rows = 77 areas x 720 hours
missing congestion index: 10.4% | missing coverage: 3.9%


,trk_congestion_index,trk_coverage
count,49673.000,53280.000
mean,1.000,0.820
std,0.058,0.297
min,0.000,0.000
25%,0.972,0.767
50%,1.000,1.000
75%,1.027,1.000
max,4.333,1.000


## 4 · Does congestion relate to crashes?

The June 2019 labels next to the index, in bands. A first look, not a model: hour of day drives both.

In [6]:
labels = pd.read_parquet("datamart/gold/label_store", filters=[("date", ">=", "2019-06-01"), ("date", "<=", "2019-07-01")])
labels = labels.sort_values("label_snapshot_date").drop_duplicates(["community_area", "hour_start"], keep="last")
joined = features.merge(labels[["community_area", "hour_start", "crash_count"]], on=["community_area", "hour_start"])
bands = pd.cut(joined["trk_congestion_index"], [0, 0.8, 0.9, 1.0, 1.1, 10], right=False)
joined.groupby(bands.cat.add_categories("no index").fillna("no index"), observed=True)["crash_count"].agg(["mean", "count"]).round(3)

,mean,count
trk_congestion_index,,
"[0.0, 0.8)",0.128,187
"[0.8, 0.9)",0.121,1173
"[0.9, 1.0)",0.222,23361
"[1.0, 1.1)",0.200,23373
"[1.1, 10.0)",0.091,1579
no index,0.065,5767


## 5 · Point in time: one value by hand

The congestion index of the Loop (area 32) at 18:00 on Friday 14 June 2019, recomputed from silver with pandas: the speeds in the hour 17:00–18:00, against the same hour on the four Fridays before.

In [7]:
T, area = datetime(2019, 6, 14, 18), 32
hour = T - timedelta(hours=1)
rows = silver.filter((F.col("community_area") == area) & F.col("speed_mean").isNotNull()
                     & F.col("hour_start").isin([hour - timedelta(days=7 * k) for k in range(5)])).toPandas()
now = rows[rows["hour_start"] == hour].set_index("segment_id")["speed_mean"]
usual = rows[rows["hour_start"] < hour].groupby("segment_id")["speed_mean"].mean()
by_hand = (now / usual).dropna().mean()
stored = features.loc[(features["community_area"] == area) & (features["hour_start"] == T), "trk_congestion_index"].item()
print(f"segments with a speed at {hour:%H:%M}: {len(now)}; by hand {by_hand:.4f}, trk_features {stored:.4f}")

segments with a speed at 17:00: 51; by hand 0.9508, trk_features 0.9508
